# DoLa + Pythia-1.4B TruthfulQA-MC on Colab\n\nThis notebook runs the low-memory GPU reproduction used in the report. It evaluates `EleutherAI/pythia-1.4b` on TruthfulQA multiple-choice with both vanilla scoring and DoLa contrastive scoring.\n\nExpected Colab free-tier setting: T4 16GB GPU.

## 1. Check GPU\n\nUse `Runtime -> Change runtime type -> T4 GPU` before running the notebook.

In [ ]:
!nvidia-smi

## 2. Clone or update the repository

In [ ]:
import os\n\nrepo_url = "https://github.com/Zhenhao526/NLPpre.git"\nrepo_dir = "/content/NLPpre"\n\nif not os.path.exists(repo_dir):\n    !git clone {repo_url} {repo_dir}\nelse:\n    %cd {repo_dir}\n    !git pull\n\n%cd {repo_dir}

## 3. Install dependencies\n\nColab normally already provides CUDA-enabled PyTorch, so the command below installs the project-level libraries and lets pip keep the existing compatible PyTorch build when possible.

In [ ]:
!python -m pip install -U -q transformers datasets accelerate sentencepiece protobuf pandas pyyaml tqdm

In [ ]:
import torch\nprint("torch:", torch.__version__)\nprint("cuda available:", torch.cuda.is_available())\nprint("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 4. Optional: HuggingFace token\n\nThe Pythia model and TruthfulQA dataset are public, so this is optional. Add a token only if downloads are rate-limited.

In [ ]:
# Optional. Do not commit real tokens.\n# from huggingface_hub import login\n# login("hf_xxx")

## 5. Smoke test: 100 TruthfulQA-MC examples\n\nThis verifies model loading, dataset loading, hidden-state extraction, DoLa scoring, and MC1/MC2/MC3 output.

In [ ]:
!python scripts/run_hf_mc_eval.py --config configs/hf_truthfulqa_pythia14_100.yaml --method all --output outputs/colab_pythia14_100.csv\n\n!cat outputs/colab_pythia14_100_summary.csv

Expected 100-example reference result from the report run:\n\n| method | MC1 | MC2 | MC3 | n |\n|---|---:|---:|---:|---:|\n| dola | 0.1400 | 0.3739 | 0.1323 | 100 |\n| vanilla | 0.1700 | 0.3533 | 0.1878 | 100 |

## 6. Full validation set: 817 TruthfulQA-MC examples\n\nOn a Colab T4 16GB GPU, the observed runtime was about 5.5 minutes after model download.

In [ ]:
!python scripts/run_hf_mc_eval.py --config configs/hf_truthfulqa_pythia14_full.yaml --method all --output outputs/colab_pythia14_full.csv\n\n!cat outputs/colab_pythia14_full_summary.csv

Reference full-validation result from the report run:\n\n| method | MC1 | MC2 | MC3 | n |\n|---|---:|---:|---:|---:|\n| dola | 0.1628 | 0.3672 | 0.1311 | 817 |\n| vanilla | 0.2081 | 0.3609 | 0.1879 | 817 |\n\nInterpretation: DoLa slightly improves MC2 on Pythia-1.4B, but lowers MC1 and MC3. This is a low-memory reproducibility check, not the paper's main LLaMA-7B/13B/33B setting.

## 7. Save outputs

In [ ]:
from google.colab import files\n\nfor path in [\n    "outputs/colab_pythia14_100_summary.csv",\n    "outputs/colab_pythia14_100.csv",\n    "outputs/colab_pythia14_full_summary.csv",\n    "outputs/colab_pythia14_full.csv",\n]:\n    files.download(path)